In [ ]:
import os
import cv2
import json
import random
import numpy as np
import face_recognition
from tqdm import tqdm

# 1. 경로 설정
video_dirs = [
    "/content/drive/MyDrive/TEST_1/data/dfdc_train_part_1",
    "/content/drive/MyDrive/TEST_1/data/dfdc_train_part_2",
    "/content/drive/MyDrive/TEST_1/data/dfdc_train_part_3",
    "/content/drive/MyDrive/TEST_1/data/dfdc_train_part_4",
]
frame_root = "/content/drive/MyDrive/project23/processed_frames"
os.makedirs(frame_root, exist_ok=True)

# 2. 메타데이터에서 REAL/FAKE 먼저 분류
all_real = []  # (video_path, part_name) 리스트
all_fake = []

for video_dir in video_dirs:
    meta_path = os.path.join(video_dir, "metadata.json")
    if not os.path.exists(meta_path):
        print(f"경고: metadata.json 없음 → {video_dir} 스킵")
        continue

    with open(meta_path, "r") as f:
        meta = json.load(f)

    part_name = os.path.basename(video_dir)

    for video_name, info in meta.items():
        video_path = os.path.join(video_dir, video_name)
        if not os.path.exists(video_path):
            continue
        if info["label"] == "REAL":
            all_real.append((video_path, part_name, video_name))
        else:
            all_fake.append((video_path, part_name, video_name))

print(f"전체 REAL 영상 수: {len(all_real)}")
print(f"전체 FAKE 영상 수: {len(all_fake)}")

# 3. REAL 수에 맞춰 FAKE 언더샘플링 (1:1)
random.shuffle(all_fake)
all_fake = all_fake[:len(all_real)]
print(f"\n언더샘플링 후 FAKE: {len(all_fake)}개 → REAL {len(all_real)}개와 1:1 매칭")

# 4. 마스킹 함수
def apply_facial_mask(frame, img_size=224):
    h, w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    face_landmarks_list = face_recognition.face_landmarks(rgb_frame)
    mask = np.zeros((h, w), dtype=np.uint8)

    if face_landmarks_list:
        for face_landmarks in face_landmarks_list:
            line_thickness = max(2, int(w * 0.015))

            if 'chin' in face_landmarks:
                pts = np.array(face_landmarks['chin'], np.int32)
                cv2.polylines(mask, [pts], False, 255, line_thickness)
            if 'top_lip' in face_landmarks:
                pts = np.array(face_landmarks['top_lip'], np.int32)
                cv2.polylines(mask, [pts], True, 255, line_thickness)
            if 'bottom_lip' in face_landmarks:
                pts = np.array(face_landmarks['bottom_lip'], np.int32)
                cv2.polylines(mask, [pts], True, 255, line_thickness)

            masked_frame = cv2.bitwise_and(frame, frame, mask=mask)

            all_pts = []
            for feature in face_landmarks.values():
                all_pts.extend(feature)
            all_pts = np.array(all_pts)

            x_min, y_min = np.min(all_pts, axis=0)
            x_max, y_max = np.max(all_pts, axis=0)

            margin_x = int((x_max - x_min) * 0.2)
            margin_y = int((y_max - y_min) * 0.2)

            x_min = max(0, x_min - margin_x)
            x_max = min(w, x_max + margin_x)
            y_min = max(0, y_min - margin_y)
            y_max = min(h, y_max + margin_y)

            cropped_face = masked_frame[y_min:y_max, x_min:x_max]

            if cropped_face.size != 0:
                return cv2.resize(cropped_face, (img_size, img_size))

    return None

# 5. 프레임 추출 함수
def extract_frames(cap, fps, start_sec, duration_sec=3, interval=8):
    start_frame = int(fps * start_sec)
    max_frames = int(fps * duration_sec)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frames = []
    idx = 0
    while idx < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % interval == 0:
            frames.append(frame)
        idx += 1

    return frames

# 6. 전처리 함수
def process_video(video_path, part_name, video_name, label):
    save_dir = os.path.join(frame_root, label, part_name, video_name[:-4])
    os.makedirs(save_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    total_sec = total_frames / fps

    # 중간 3초 시도
    mid_start = max(0, (total_sec / 2) - 1.5)
    frames = extract_frames(cap, fps, start_sec=mid_start, duration_sec=3, interval=8)

    saved = 0
    for frame in frames:
        processed = apply_facial_mask(frame, img_size=224)
        if processed is not None:
            cv2.imwrite(f"{save_dir}/frame_{saved}.jpg", processed)
            saved += 1

    # fallback: 앞 3초
    if saved == 0:
        frames = extract_frames(cap, fps, start_sec=0, duration_sec=3, interval=8)
        for frame in frames:
            processed = apply_facial_mask(frame, img_size=224)
            if processed is not None:
                cv2.imwrite(f"{save_dir}/frame_{saved}.jpg", processed)
                saved += 1

    cap.release()
    return saved

# 7. REAL 전처리
print("\nREAL 영상 전처리 시작...")
for video_path, part_name, video_name in tqdm(all_real, desc="REAL"):
    process_video(video_path, part_name, video_name, "REAL")

# 8. FAKE 전처리
print("\nFAKE 영상 전처리 시작...")
for video_path, part_name, video_name in tqdm(all_fake, desc="FAKE"):
    process_video(video_path, part_name, video_name, "FAKE")

print(f"\n전체 전처리 완료! 저장 경로: {frame_root}")